# Notebook 3: Feature Engineering

Builds temporal lag/rolling features, spatial neighbor features, and removes leakage features.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.feature_engineering import (
    create_temporal_features, impute_temporal_features,
    remove_leakage_features, create_spatial_features
)
from src.spatial_features import extract_basin_coordinates, build_topology, plot_basin_map

In [ ]:
df = pd.read_csv('../data/master_with_floodflags.csv')
print('Loaded:', df.shape)

In [ ]:
# ── Temporal features ─────────────────────────────────────────
df_temporal = create_temporal_features(df)
df_temporal = impute_temporal_features(df_temporal)
df_temporal = remove_leakage_features(df_temporal)
print('After temporal features:', df_temporal.shape)

In [ ]:
# ── Extract real basin coordinates from shapefile ────────────
coords_df = extract_basin_coordinates(
    '../data/Indus_SubBasins_Pakistan.shp',
    out_path='../data/your_basin_coordinates.csv'
)
coords_df.head()

In [ ]:
# ── Spatial features ──────────────────────────────────────────
df_spatial = create_spatial_features(df_temporal, coords_df, k_neighbors=5)
df_spatial.to_csv('../data/flood_data_with_spatial_features.csv', index=False)
print('Final shape:', df_spatial.shape)
print('Features:', df_spatial.shape[1])
df_spatial.head()

In [ ]:
# ── Feature correlation heatmap ───────────────────────────────
numeric_cols = df_spatial.select_dtypes(include='number').columns.tolist()
exclude = ['HYBAS_ID','year','month']
feat_cols = [c for c in numeric_cols if c not in exclude][:20]  # top 20

plt.figure(figsize=(14, 10))
corr = df_spatial[feat_cols + ['Flood_Flag']].corr()
sns_mask = np.zeros_like(corr, dtype=bool)
plt.title('Feature Correlation Matrix (top 20 features + target)')
import seaborn as sns
sns.heatmap(corr, mask=np.triu(corr), cmap='coolwarm', center=0,
            annot=False, fmt='.2f', linewidths=0.5)
plt.tight_layout()
plt.savefig('../results/feature_correlation.png', dpi=150)
plt.show()